# TurboQuant KV-cache compression

TurboQuant is not a weight quantizer; it compresses the runtime KV cache. The paper reports "absolute quality neutrality with 3.5 bits per channel" for KV-cache quantization.

Sources:
- Paper: https://arxiv.org/abs/2504.19874
- Community PyTorch implementation: https://github.com/tonbistudio/turboquant-pytorch
- Community implementation: https://github.com/OnlyTerp/turboquant

This notebook captures a real Hugging Face `past_key_values` cache, applies a self-contained TurboQuant-style random-rotation plus scalar quantization reference path, saves a cache artifact, and reports theoretical packed memory. Use the community repos or future vLLM/llama.cpp integrations for production kernels.

In [ ]:
!pip -q install -U transformers accelerate sentencepiece safetensors

In [ ]:

import json
import math
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM


def bytes_to_mib(n):
    return n / (1024 ** 2)


def make_random_orthogonal(dim, device, dtype, seed=1234):
    g = torch.Generator(device=device)
    g.manual_seed(seed)
    a = torch.randn(dim, dim, generator=g, device=device, dtype=torch.float32)
    q, _ = torch.linalg.qr(a)
    return q.to(device=device, dtype=dtype)


def quantize_rotated_vectors(x, bits, rotation):
    orig_shape = x.shape
    dim = orig_shape[-1]
    flat = x.reshape(-1, dim).to(rotation.dtype)
    rotated = flat @ rotation
    qmax = (2 ** (bits - 1)) - 1
    scale = rotated.abs().amax(dim=-1, keepdim=True).clamp(min=1e-6) / qmax
    q = torch.round(rotated / scale).clamp(-qmax, qmax).to(torch.int8)
    deq = (q.to(rotation.dtype) * scale) @ rotation.T
    return {
        "q": q.cpu(),
        "scale": scale.cpu().to(torch.float16),
        "dequantized": deq.reshape(orig_shape).to(x.dtype),
    }


def kv_cache_bytes(past_key_values):
    total = 0
    for key, value in past_key_values:
        total += key.numel() * key.element_size()
        total += value.numel() * value.element_size()
    return total


def cosine_report(original, reconstructed):
    a = original.reshape(-1, original.shape[-1]).float()
    b = reconstructed.reshape(-1, reconstructed.shape[-1]).float()
    cos = F.cosine_similarity(a, b, dim=-1)
    mse = F.mse_loss(b, a).item()
    return float(cos.mean().item()), float(cos.min().item()), float(mse)


def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(json.dumps(payload, indent=2))


In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = Path("/content/quant_outputs/turboquant_kv")
PROMPT = "TurboQuant compresses the key value cache for long context inference. " * 32
BITS = 3
INCLUDE_QJL_RESIDUAL_BITS = True

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to(device)
model.eval()

inputs = tokenizer(PROMPT, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs, use_cache=True)
past = outputs.past_key_values

head_dim = past[0][0].shape[-1]
rotation = make_random_orthogonal(head_dim, device, past[0][0].dtype)

artifact = {"method": "turboquant_reference", "bits": BITS, "layers": []}
reports = []
packed_bits = 0
for layer_idx, (key, value) in enumerate(past):
    qk = quantize_rotated_vectors(key, BITS, rotation)
    qv = quantize_rotated_vectors(value, BITS, rotation)
    k_mean, k_min, k_mse = cosine_report(key, qk["dequantized"].to(device))
    v_mean, v_min, v_mse = cosine_report(value, qv["dequantized"].to(device))
    n_values = key.numel() + value.numel()
    n_vectors = key.reshape(-1, head_dim).shape[0] + value.reshape(-1, head_dim).shape[0]
    packed_bits += n_values * BITS
    packed_bits += n_vectors * 16  # per-vector fp16 scale
    if INCLUDE_QJL_RESIDUAL_BITS:
        packed_bits += n_values  # one residual sign bit per coordinate
    artifact["layers"].append({
        "key_q": qk["q"],
        "key_scale": qk["scale"],
        "value_q": qv["q"],
        "value_scale": qv["scale"],
    })
    reports.append({
        "layer": layer_idx,
        "key_cosine_mean": k_mean,
        "key_cosine_min": k_min,
        "key_mse": k_mse,
        "value_cosine_mean": v_mean,
        "value_cosine_min": v_min,
        "value_mse": v_mse,
    })

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(artifact, OUTPUT_DIR / "compressed_kv_cache.pt")

fp_bytes = kv_cache_bytes(past)
packed_bytes = math.ceil(packed_bits / 8)
metrics = {
    "method": "turboquant_reference_kv_cache",
    "model_id": MODEL_ID,
    "bits": BITS,
    "qjl_residual_bits_included": INCLUDE_QJL_RESIDUAL_BITS,
    "fp_cache_mib": bytes_to_mib(fp_bytes),
    "theoretical_packed_cache_mib": bytes_to_mib(packed_bytes),
    "compression_ratio": fp_bytes / packed_bytes,
    "artifact_path": str(OUTPUT_DIR / "compressed_kv_cache.pt"),
    "reports": reports,
}
save_json(OUTPUT_DIR / "metrics.json", metrics)

In [ ]:
# Optional: clone a community implementation for deeper experiments.
# !git clone https://github.com/tonbistudio/turboquant-pytorch /content/turboquant-pytorch
# %cd /content/turboquant-pytorch
# !pip -q install -r requirements.txt
# !python validate.py